# AI-Powered SOC Assistant

This Colab notebook is a thin, reproducible driver for the repository code. It intentionally does **not** duplicate preprocessing, training, SHAP, or ticket-generation logic. The default workflow uses the deterministic template provider and makes no LLM or threat-intelligence API calls.

For an archived experiment, set `AI_SOC_REPO_REF` to a commit SHA before running the next cell. The default is the latest `main` branch.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/opandey1/AI-SOC-Assistant.git"
REPO_REF = os.environ.get("AI_SOC_REPO_REF", "main")
REPO_DIR = Path("/content/AI-SOC-Assistant")
VENV_DIR = Path("/content/ai-soc-venv")

if sys.version_info[:2] not in {(3, 10), (3, 11), (3, 12)}:
    raise RuntimeError("This project supports Python 3.10 through 3.12.")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)],
        check=True,
    )
subprocess.run(
    ["git", "fetch", "--depth", "1", "origin", REPO_REF],
    cwd=REPO_DIR,
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", "FETCH_HEAD"],
    cwd=REPO_DIR,
    check=True,
)
os.chdir(REPO_DIR)
PROJECT_PYTHON = VENV_DIR / "bin" / "python"
if not PROJECT_PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
subprocess.run(
    [str(PROJECT_PYTHON), "-m", "pip", "install", "-r", "requirements.txt"],
    check=True,
)
subprocess.run([str(PROJECT_PYTHON), "-m", "pip", "check"], check=True)
print(
    f"Ready: {REPO_DIR} at {REPO_REF} on Python {sys.version.split()[0]} "
    f"with isolated environment {VENV_DIR}"
)

## Upload and verify NSL-KDD

Download `KDDTrain+.txt` and `KDDTest+.txt` through the [University of New Brunswick NSL-KDD page](https://www.unb.ca/cic/datasets/nsl.html), review its terms, and upload both files below. The cell prints SHA-256 checksums so the input provenance can be recorded with the run. Reference checksums are documented in `data/README.md`.

In [ ]:
from google.colab import files
import hashlib

DATA_DIR = REPO_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)
required_files = ("KDDTrain+.txt", "KDDTest+.txt")
missing = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing:
    print("Upload: " + ", ".join(missing))
    uploaded = files.upload()
    for name, contents in uploaded.items():
        destination = DATA_DIR / Path(name).name
        destination.write_bytes(contents)

still_missing = [name for name in required_files if not (DATA_DIR / name).exists()]
if still_missing:
    raise FileNotFoundError("Missing required files: " + ", ".join(still_missing))

for name in required_files:
    path = DATA_DIR / name
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{name}: {path.stat().st_size:,} bytes, sha256={digest}")

## Confirm the shared ingestion path

This imports `src.ingest` directly; changes made in the production package therefore apply to this notebook without maintaining a second implementation.

In [ ]:
summary_program = """
from pathlib import Path
from src.ingest import dataset_summary, load_nsl_kdd
data_dir = Path('data')
dataset = load_nsl_kdd(data_dir / 'KDDTrain+.txt', data_dir / 'KDDTest+.txt')
print(dataset_summary(dataset))
"""
subprocess.run(
    [str(PROJECT_PYTHON), "-c", summary_program],
    cwd=REPO_DIR,
    check=True,
)

## Run the offline pipeline

The command trains the models, selects a flagged connection, builds its SHAP explanation, and renders a deterministic SOC ticket. It does not require an API key or an LLM server.

In [ ]:
subprocess.run(
    [str(PROJECT_PYTHON), "-m", "src.pipeline", "--no-llm"],
    cwd=REPO_DIR,
    check=True,
)

## Optional evaluation

Set the switch below to regenerate the hold-out metrics and visual artifacts. Keep it false for a quicker demonstration. Use the separate `--use-test-set` command described in the README when evaluating cross-distribution generalization on `KDDTest+`.

In [ ]:
RUN_EVALUATION = False

if RUN_EVALUATION:
    subprocess.run(
        [str(PROJECT_PYTHON), "-m", "src.evaluate"],
        cwd=REPO_DIR,
        check=True,
    )
else:
    print("Evaluation skipped. Set RUN_EVALUATION=True to run it.")

## Privacy boundary for optional integrations

The notebook intentionally stops at offline template mode. OpenAI or Anthropic providers transmit the prompt and connection evidence to an external service. Threat-intelligence tools are separately disabled by default; setting `SOC_ENABLE_THREAT_INTEL=true` permits outbound AbuseIPDB and NVD requests that may disclose a source IP or service name. Use either option only with authorization for the telemetry involved. Ollama is best run on your own machine or controlled infrastructure rather than installed through an unreviewed notebook script.